<a href="https://colab.research.google.com/github/Ag3-S/Neural-Networks-Lab/blob/main/Experiment2_NNLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt


In [3]:

class BasicNeuron:
    """
    A basic artificial neuron implementation that mimics biological neurons.

    The neuron receives inputs, applies weights, adds bias, and produces an output
    through an activation function - just like neurons in our brain!
    """

    def __init__(self, num_inputs, activation_function='sigmoid'):
        """
        Initialize the neuron with random weights and bias.

        Args:
            num_inputs: Number of input connections to this neuron
            activation_function: Type of activation function ('sigmoid', 'relu', 'tanh', 'linear')
        """
        # Initialize weights randomly between -1 and 1
        # Each input gets its own weight - this determines how important each input is
        self.weights = np.random.uniform(-1, 1, num_inputs)

        # Initialize bias - this shifts the activation function left or right
        # Bias helps the neuron fire even when inputs are small
        self.bias = np.random.uniform(-1, 1)

        # Store the activation function type
        if activation_function not in ['sigmoid', 'relu', 'tanh', 'linear']:
            raise ValueError(f"Unsupported activation function: {activation_function}")
        self.activation_function = activation_function

        # Store the number of inputs for validation
        self.num_inputs = num_inputs

        # print(f"Neuron created with {num_inputs} inputs")
        # print(f"Initial weights: {self.weights}")
        # print(f"Initial bias: {self.bias}")
        # print(f"Activation function: {activation_function}")

    def sigmoid(self, x):
        """
        Sigmoid activation function: f(x) = 1 / (1 + e^(-x))

        - Outputs values between 0 and 1
        - Smooth, differentiable curve
        - Good for binary classification problems
        """
        # Clip x to prevent overflow in exponential
        x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        """Derivative of the sigmoid function."""
        # Derivative of sigmoid(x) is sigmoid(x) * (1 - sigmoid(x))
        # We can use the output of the sigmoid function itself for this
        s = self.sigmoid(x)
        return s * (1 - s)

    def relu(self, x):
        """ReLU activation function: f(x) = max(0, x)"""
        return np.maximum(0, x)

    def relu_derivative(self, x):
        """Derivative of the ReLU function."""
        return np.where(x > 0, 1, 0)

    def tanh(self, x):
        """Tanh activation function: f(x) = tanh(x)"""
        return np.tanh(x)

    def tanh_derivative(self, x):
        """Derivative of the Tanh function."""
        # Derivative of tanh(x) is 1 - tanh(x)^2
        return 1 - np.tanh(x)**2

    def linear(self, x):
        """Linear activation function: f(x) = x"""
        return x

    def linear_derivative(self, x):
        """Derivative of the Linear function."""
        return 1

    def get_activation_derivative(self, z):
        """Returns the derivative of the chosen activation function for a given net input z."""
        if self.activation_function == 'sigmoid':
            return self.sigmoid_derivative(z)
        elif self.activation_function == 'relu':
            return self.relu_derivative(z)
        elif self.activation_function == 'tanh':
            return self.tanh_derivative(z)
        elif self.activation_function == 'linear':
            return self.linear_derivative(z)
        else:
            raise ValueError(f"Derivative not implemented for {self.activation_function}")


    def forward(self, inputs):

        # Convert inputs to numpy array for easier computation
        inputs = np.array(inputs)

        # Validate input size
        if len(inputs) != self.num_inputs:
            raise ValueError(f"Expected {self.num_inputs} inputs, got {len(inputs)}")

        # Step 1 & 2: Weighted sum (dot product of inputs and weights)
        # This is like: w1*x1 + w2*x2 + w3*x3 + ... + wn*xn
        weighted_sum = np.dot(inputs, self.weights)

        # Step 3: Add bias
        # Bias allows the neuron to fire even when inputs are zero
        z = weighted_sum + self.bias

        # Step 4: Apply activation function
        if self.activation_function == 'sigmoid':
            output = self.sigmoid(z)
        elif self.activation_function == 'relu':
            output = self.relu(z)
        elif self.activation_function == 'tanh':
            output = self.tanh(z)
        elif self.activation_function == 'linear':
            output = self.linear(z)
        else:
            raise ValueError(f"Unknown activation function: {self.activation_function}")

        # Store intermediate values for educational purposes
        self.last_inputs = inputs
        self.last_weighted_sum = weighted_sum
        self.last_z = z # Store z (net input before activation) for derivative calculation
        self.last_output = output

        return output

    def update_weights(self, new_weights, new_bias=None):
        if len(new_weights) != self.num_inputs:
            raise ValueError(f"Expected {self.num_inputs} weights, got {len(new_weights)}")

        self.weights = np.array(new_weights)

        if new_bias is not None:
            self.bias = new_bias

        # print(f"Weights updated to: {self.weights}")
        # print(f"Bias updated to: {self.bias}")

    def get_details(self, verbose=True):
        if hasattr(self, 'last_inputs'):
            if verbose:
                print("\n--- Neuron Computation Details ---")
                print(f"Inputs: {self.last_inputs}")
                print(f"Weights: {self.weights}")
                print(f"Weighted sum: {self.last_weighted_sum:.4f}")
                print(f"Bias: {self.bias:.4f}")
                print(f"z (weighted sum + bias): {self.last_z:.4f}")
                print(f"Final output: {self.last_output:.4f}")
        else:
            if verbose:
                print("No computation has been performed yet!")


 --- 1. Hebbian Learning ---
#### Hebbian learning is one of the oldest and simplest learning rules.
#### It states that if two neurons on either side of a synapse are
#### simultaneously active, then the strength of that synapse is increased.
#### In its simplest form, the weight update is proportional to the product
#### of the pre-synaptic and post-synaptic activations.

In [17]:
def hebbian_learning(inputs, outputs, learning_rate=1.0):

    num_inputs = inputs.shape[1]
    # Hebbian learning typically doesn't use a complex activation function for its rule,
    # so we use 'linear' in BasicNeuron and manage the output directly.
    neuron = BasicNeuron(num_inputs, activation_function='linear')
    # Initialize weights to zeros as per traditional Hebbian setup for simplicity in this demo
    neuron.update_weights(np.zeros(num_inputs), 0.0)

    num_patterns = inputs.shape[0]

    print(f"\n--- Hebbian Learning ---")
    print(f"Initial Weights: {neuron.weights}, Initial Bias: {neuron.bias}")

    for i in range(num_patterns):
        x = inputs[i]
        y_desired = outputs[i]

        # Hebbian update rule: delta_w_i = learning_rate * x_i * y_desired
        # Here, we assume the output 'y_desired' is the post-synaptic activity.
        delta_weights = learning_rate * x * y_desired
        new_weights = neuron.weights + delta_weights
        new_bias = neuron.bias # Hebbian rule typically doesn't include bias update in this form

        neuron.update_weights(new_weights, new_bias)

        print(f"Pattern {i+1}: Input={x}, Desired Output={y_desired}")
        print(f"  Delta Weights: {delta_weights}")
        print(f"  Updated Weights: {neuron.weights}, Updated Bias: {neuron.bias}")

    print(f"Final Weights (Hebbian): {neuron.weights}, Final Bias: {neuron.bias}")
    return neuron.weights, neuron.bias


#### Example for Hebbian Learning: Simple pattern association
#### Let's try to learn a pattern where input [1, 1] should activate, others not.
#### Note: Hebbian learning is often used for associative memory, not direct classification.
#### It tends to learn correlations.

In [18]:
hebbian_inputs = np.array([
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 0]
])
hebbian_outputs = np.array([
    0,  # For [1, 0]
    0,  # For [0, 1]
    1,  # For [1, 1] - this pattern should strengthen weights
    0   # For [0, 0]
])
hebbian_learning(hebbian_inputs, hebbian_outputs)



--- Hebbian Learning ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Pattern 1: Input=[1 0], Desired Output=0
  Delta Weights: [0. 0.]
  Updated Weights: [0. 0.], Updated Bias: 0.0
Pattern 2: Input=[0 1], Desired Output=0
  Delta Weights: [0. 0.]
  Updated Weights: [0. 0.], Updated Bias: 0.0
Pattern 3: Input=[1 1], Desired Output=1
  Delta Weights: [1. 1.]
  Updated Weights: [1. 1.], Updated Bias: 0.0
Pattern 4: Input=[0 0], Desired Output=0
  Delta Weights: [0. 0.]
  Updated Weights: [1. 1.], Updated Bias: 0.0
Final Weights (Hebbian): [1. 1.], Final Bias: 0.0


(array([1., 1.]), 0.0)

In [24]:
weights, bias = hebbian_learning(hebbian_inputs, hebbian_outputs)

testN = BasicNeuron(2, activation_function='sigmoid')
testN.update_weights(weights, bias)

testN.forward([0,1])


--- Hebbian Learning ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Pattern 1: Input=[1 0], Desired Output=0
  Delta Weights: [0. 0.]
  Updated Weights: [0. 0.], Updated Bias: 0.0
Pattern 2: Input=[0 1], Desired Output=0
  Delta Weights: [0. 0.]
  Updated Weights: [0. 0.], Updated Bias: 0.0
Pattern 3: Input=[1 1], Desired Output=1
  Delta Weights: [1. 1.]
  Updated Weights: [1. 1.], Updated Bias: 0.0
Pattern 4: Input=[0 0], Desired Output=0
  Delta Weights: [0. 0.]
  Updated Weights: [1. 1.], Updated Bias: 0.0
Final Weights (Hebbian): [1. 1.], Final Bias: 0.0


np.float64(0.7310585786300049)

#### --- 2. Perceptron Learning Rule ---
#### The Perceptron learning rule is an algorithm for supervised learning of binary classifiers.
#### It is used for linearly separable data. The rule updates weights only when a
#### misclassification occurs.

In [31]:
def activation_step(net_input):
    """Step activation function for perceptron."""
    return 1 if net_input >= 0 else 0

def perceptron_learning(inputs, outputs, learning_rate=0.1, epochs=100):
    """
    Demonstrates the Perceptron learning rule using BasicNeuron class.
    Assumes binary inputs (0 or 1) and binary outputs (0 or 1).

    Args:
        inputs (np.array): A 2D array where each row is an input pattern.
        outputs (np.array): A 1D array of desired outputs for each input pattern.
        learning_rate (float): The learning rate (eta).
        epochs (int): Number of training iterations.
    """
    num_inputs = inputs.shape[1]
    # Use 'linear' activation in BasicNeuron and apply step function externally
    # because BasicNeuron's built-in activations are continuous.
    neuron = BasicNeuron(num_inputs, activation_function='linear')

    num_patterns = inputs.shape[0]

    print(f"\n--- Perceptron Learning Rule ---")
    print(f"Initial Weights: {neuron.weights}, Initial Bias: {neuron.bias}")

    for epoch in range(epochs):
        errors = 0
        for i in range(num_patterns):
            x = inputs[i]
            y_desired = outputs[i]

            # Calculate net input using neuron's forward pass (before step activation)
            # neuron.forward(x) will return the linear output (z)
            net_input_raw = neuron.forward(x)

            # Get actual output by applying the step activation
            y_actual = activation_step(net_input_raw)

            # Calculate error
            error = y_desired - y_actual

            if error != 0: # Update weights only on misclassification
                # Perceptron update rule: delta_w_i = learning_rate * error * x_i
                # delta_bias = learning_rate * error
                new_weights = neuron.weights + learning_rate * error * x
                new_bias = neuron.bias + learning_rate * error
                neuron.update_weights(new_weights, new_bias)
                errors += 1

        # print(f"Epoch {epoch+1}: Errors = {errors}, Weights = {neuron.weights}, Bias = {neuron.bias}")
        if errors == 0:
            print(f"Converged at Epoch {epoch+1}.")
            break
    else:
        print(f"Did not converge within {epochs} epochs.")

    print(f"Final Weights (Perceptron): {neuron.weights}, Final Bias: {neuron.bias}")

    # Test the trained perceptron
    print("\n--- Perceptron Test ---")
    for i in range(num_patterns):
        x = inputs[i]
        net_input_raw = neuron.forward(x)
        y_predicted = activation_step(net_input_raw)
        print(f"Input: {x}, Desired: {outputs[i]}, Predicted: {y_predicted}")


In [32]:
# Example for Perceptron Learning: AND gate
# This is a linearly separable problem.
and_inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
and_outputs = np.array([0, 0, 0, 1])
perceptron_learning(and_inputs, and_outputs, learning_rate=0.1, epochs=100)



--- Perceptron Learning Rule ---
Initial Weights: [-0.78108051 -0.33228518], Initial Bias: 0.549603766713946
Converged at Epoch 14.
Final Weights (Perceptron): [0.21891949 0.06771482], Final Bias: -0.25039623328605404

--- Perceptron Test ---
Input: [0 0], Desired: 0, Predicted: 0
Input: [0 1], Desired: 0, Predicted: 0
Input: [1 0], Desired: 0, Predicted: 0
Input: [1 1], Desired: 1, Predicted: 1


In [33]:
# Example for Perceptron Learning: XOR gate (Non-linearly separable)
# This will demonstrate that a single perceptron cannot solve XOR.
xor_inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
xor_outputs = np.array([0, 1, 1, 0])
# Uncomment to see Perceptron fail on XOR
perceptron_learning(xor_inputs, xor_outputs, learning_rate=0.1, epochs=100)



--- Perceptron Learning Rule ---
Initial Weights: [-0.10159991 -0.61138099], Initial Bias: -0.04953753092487245
Did not converge within 100 epochs.
Final Weights (Perceptron): [-0.10159991 -0.21138099], Final Bias: 0.05046246907512755

--- Perceptron Test ---
Input: [0 0], Desired: 0, Predicted: 1
Input: [0 1], Desired: 1, Predicted: 0
Input: [1 0], Desired: 1, Predicted: 0
Input: [1 1], Desired: 0, Predicted: 0


#### --- 3. Delta Rule (Widrow-Hoff Rule / LMS Rule) ---
#### The Delta Rule is a gradient descent learning rule for updating the weights
#### of artificial neurons in a single-layer feedforward network. It's a generalization
#### of the perceptron learning rule and is often used with continuous activation
#### functions (like linear or sigmoid) to minimize the mean squared error (MSE).


In [34]:
def delta_rule_learning(inputs, outputs, learning_rate=0.1, epochs=100, activation_func='linear'):
    """
    Demonstrates the Delta learning rule using BasicNeuron class.

    Args:
        inputs (np.array): A 2D array where each row is an input pattern.
        outputs (np.array): A 1D array of desired outputs for each input pattern.
        learning_rate (float): The learning rate (eta).
        epochs (int): Number of training iterations.
        activation_func (str): 'linear' or 'sigmoid' for the activation function.
    """
    num_inputs = inputs.shape[1]
    neuron = BasicNeuron(num_inputs, activation_function=activation_func)

    num_patterns = inputs.shape[0]

    print(f"\n--- Delta Rule Learning ({activation_func.capitalize()} Activation) ---")
    print(f"Initial Weights: {neuron.weights}, Initial Bias: {neuron.bias}")

    for epoch in range(epochs):
        total_error = 0
        for i in range(num_patterns):
            x = inputs[i]
            y_desired = outputs[i]

            # Calculate actual output using neuron's forward pass
            y_actual = neuron.forward(x)

            # Calculate error
            error = y_desired - y_actual
            total_error += 0.5 * (error ** 2) # Sum of squared errors

            # Get derivative of activation function (f'(net_input))
            # The last_z attribute stores the net input (z) before activation
            derivative_activation = neuron.get_activation_derivative(neuron.last_z)

            # Delta Rule update: delta_w_i = learning_rate * error * x_i * f'(net_input)
            new_weights = neuron.weights + learning_rate * error * x * derivative_activation
            new_bias = neuron.bias + learning_rate * error * derivative_activation
            neuron.update_weights(new_weights, new_bias)

        print(f"Epoch {epoch+1}: Total Squared Error = {total_error:.4f}, Weights = {neuron.weights}, Bias = {neuron.bias:.4f}")
        # Stop if error is very small
        if total_error < 0.001:
            print(f"Converged at Epoch {epoch+1}.")
            break
    else:
        print(f"Did not converge within {epochs} epochs.")

    print(f"Final Weights (Delta Rule): {neuron.weights}, Final Bias: {neuron.bias}")

    # Test the trained network
    print(f"\n--- Delta Rule Test ({activation_func.capitalize()} Activation) ---")
    for i in range(num_patterns):
        x = inputs[i]
        y_predicted = neuron.forward(x)
        print(f"Input: {x}, Desired: {outputs[i]:.4f}, Predicted: {y_predicted:.4f}")


In [35]:
# Example for Delta Rule: Simple linear regression type problem
# Input x, desired output 2x + 1 (approximately)
delta_inputs = np.array([
    [0.1], [0.2], [0.3], [0.4], [0.5],
    [0.6], [0.7], [0.8], [0.9], [1.0]
])
delta_outputs = np.array([
    0.3, 0.5, 0.7, 0.9, 1.1,
    1.3, 1.5, 1.7, 1.9, 2.1
])

# Linear activation
delta_rule_learning(delta_inputs, delta_outputs, learning_rate=0.05, epochs=1000, activation_func='linear')



--- Delta Rule Learning (Linear Activation) ---
Initial Weights: [0.59498967], Initial Bias: 0.31770938659641956
Epoch 1: Total Squared Error = 1.5539, Weights = [0.76658193], Bias = 0.5497
Epoch 2: Total Squared Error = 0.7138, Weights = [0.86952992], Bias = 0.6557
Epoch 3: Total Squared Error = 0.5169, Weights = [0.93742195], Bias = 0.6986
Epoch 4: Total Squared Error = 0.4634, Weights = [0.98717086], Bias = 0.7101
Epoch 5: Total Squared Error = 0.4377, Weights = [1.02730056], Bias = 0.7062
Epoch 6: Total Squared Error = 0.4153, Weights = [1.06211445], Bias = 0.6947
Epoch 7: Total Squared Error = 0.3928, Weights = [1.09379239], Bias = 0.6798
Epoch 8: Total Squared Error = 0.3701, Weights = [1.12344552], Bias = 0.6635
Epoch 9: Total Squared Error = 0.3481, Weights = [1.1516469], Bias = 0.6466
Epoch 10: Total Squared Error = 0.3269, Weights = [1.17869849], Bias = 0.6299
Epoch 11: Total Squared Error = 0.3068, Weights = [1.20476544], Bias = 0.6134
Epoch 12: Total Squared Error = 0.2878

In [36]:
#Example for Delta Rule with Sigmoid: Approximating a binary output with continuous values
# (Similar to AND gate, but outputs will be continuous between 0 and 1)
sigmoid_inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
sigmoid_outputs = np.array([0, 0, 0, 1]) # Target values are binary, but output will be continuous

delta_rule_learning(sigmoid_inputs, sigmoid_outputs, learning_rate=0.1, epochs=5000, activation_func='sigmoid')


Streaming output truncated to the last 5000 lines.
Epoch 9: Total Squared Error = 0.5506, Weights = [ 0.18947914 -0.28139778], Bias = 0.1443
Epoch 10: Total Squared Error = 0.5433, Weights = [ 0.18797749 -0.2803186 ], Bias = 0.1179
Epoch 11: Total Squared Error = 0.5364, Weights = [ 0.18674249 -0.27890275], Bias = 0.0922
Epoch 12: Total Squared Error = 0.5298, Weights = [ 0.18576588 -0.27716591], Bias = 0.0670
Epoch 13: Total Squared Error = 0.5235, Weights = [ 0.18503939 -0.2751234 ], Bias = 0.0424
Epoch 14: Total Squared Error = 0.5174, Weights = [ 0.18455475 -0.27279011], Bias = 0.0184
Epoch 15: Total Squared Error = 0.5116, Weights = [ 0.18430379 -0.2701804 ], Bias = -0.0051
Epoch 16: Total Squared Error = 0.5060, Weights = [ 0.18427844 -0.26730814], Bias = -0.0281
Epoch 17: Total Squared Error = 0.5007, Weights = [ 0.18447075 -0.26418665], Bias = -0.0506
Epoch 18: Total Squared Error = 0.4955, Weights = [ 0.18487297 -0.26082867], Bias = -0.0726
Epoch 19: Total Squared Error = 0.49

### Experiment with Activation Functions:

Try different combinations of activation functions for the hidden and output layers (e.g., sigmoid for both, relu for hidden and sigmoid for output). Observe how this affects convergence and performance.

Correlation Learning Law

The Correlation Learning Law updates the weights according to the correlation between each input and its desired output. Unlike the Delta Rule, it does not calculate an error or use the neuron's current output for learning.

In [38]:
def correlation_learning(inputs, outputs, learning_rate=1.0):
    """
    Demonstrates the Correlation Learning Law using BasicNeuron class.

    The correlation learning rule updates weights according to:
        delta_w_i = learning_rate * x_i * target

    and the bias according to:
        delta_bias = learning_rate * target

    Args:
        inputs (np.array): A 2D array where each row is an input pattern.
        outputs (np.array): A 1D array of desired/target outputs.
        learning_rate (float): The learning rate (eta).

    Returns:
        tuple: Final weights and bias.
    """

    num_inputs = inputs.shape[1]

    # Correlation learning does not require the current
    # neuron output for calculating the weight update.
    neuron = BasicNeuron(num_inputs, activation_function='linear')

    # Initialize weights and bias to zero
    neuron.update_weights(np.zeros(num_inputs), 0.0)

    num_patterns = inputs.shape[0]

    print("\n--- Correlation Learning Law ---")
    print(f"Initial Weights: {neuron.weights}, Initial Bias: {neuron.bias}")

    for i in range(num_patterns):
        x = inputs[i]
        y_desired = outputs[i]

        # Correlation Learning Rule:
        # delta_w_i = learning_rate * x_i * target
        delta_weights = learning_rate * x * y_desired

        # Bias is treated as a weight connected to constant input 1
        delta_bias = learning_rate * y_desired

        # Update weights and bias
        new_weights = neuron.weights + delta_weights
        new_bias = neuron.bias + delta_bias

        neuron.update_weights(new_weights, new_bias)

        print(f"Pattern {i+1}: Input={x}, Desired Output={y_desired}")
        print(f"  Delta Weights: {delta_weights}")
        print(f"  Delta Bias: {delta_bias}")
        print(f"  Updated Weights: {neuron.weights}")
        print(f"  Updated Bias: {neuron.bias}")

    print(f"\nFinal Weights (Correlation): {neuron.weights}")
    print(f"Final Bias: {neuron.bias}")

    return neuron.weights, neuron.bias

In [39]:
# Example for Correlation Learning:
# Bipolar AND gate

correlation_inputs = np.array([
    [-1, -1],
    [-1,  1],
    [ 1, -1],
    [ 1,  1]
])

correlation_outputs = np.array([
    -1,   # [-1, -1]
    -1,   # [-1,  1]
    -1,   # [ 1, -1]
     1    # [ 1,  1]
])

correlation_learning(
    correlation_inputs,
    correlation_outputs,
    learning_rate=1.0
)


--- Correlation Learning Law ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Pattern 1: Input=[-1 -1], Desired Output=-1
  Delta Weights: [1. 1.]
  Delta Bias: -1.0
  Updated Weights: [1. 1.]
  Updated Bias: -1.0
Pattern 2: Input=[-1  1], Desired Output=-1
  Delta Weights: [ 1. -1.]
  Delta Bias: -1.0
  Updated Weights: [2. 0.]
  Updated Bias: -2.0
Pattern 3: Input=[ 1 -1], Desired Output=-1
  Delta Weights: [-1.  1.]
  Delta Bias: -1.0
  Updated Weights: [1. 1.]
  Updated Bias: -3.0
Pattern 4: Input=[1 1], Desired Output=1
  Delta Weights: [1. 1.]
  Delta Bias: 1.0
  Updated Weights: [2. 2.]
  Updated Bias: -2.0

Final Weights (Correlation): [2. 2.]
Final Bias: -2.0


(array([2., 2.]), np.float64(-2.0))

In [40]:
# Train the correlation learner
weights, bias = correlation_learning(
    correlation_inputs,
    correlation_outputs,
    learning_rate=1.0
)

# Create a neuron for testing
correlation_neuron = BasicNeuron(
    2,
    activation_function='linear'
)

correlation_neuron.update_weights(weights, bias)

print("\n--- Correlation Learning Test ---")

for i in range(len(correlation_inputs)):
    x = correlation_inputs[i]
    y_predicted = correlation_neuron.forward(x)

    print(
        f"Input: {x}, "
        f"Desired: {correlation_outputs[i]}, "
        f"Net Output: {y_predicted:.4f}"
    )


--- Correlation Learning Law ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Pattern 1: Input=[-1 -1], Desired Output=-1
  Delta Weights: [1. 1.]
  Delta Bias: -1.0
  Updated Weights: [1. 1.]
  Updated Bias: -1.0
Pattern 2: Input=[-1  1], Desired Output=-1
  Delta Weights: [ 1. -1.]
  Delta Bias: -1.0
  Updated Weights: [2. 0.]
  Updated Bias: -2.0
Pattern 3: Input=[ 1 -1], Desired Output=-1
  Delta Weights: [-1.  1.]
  Delta Bias: -1.0
  Updated Weights: [1. 1.]
  Updated Bias: -3.0
Pattern 4: Input=[1 1], Desired Output=1
  Delta Weights: [1. 1.]
  Delta Bias: 1.0
  Updated Weights: [2. 2.]
  Updated Bias: -2.0

Final Weights (Correlation): [2. 2.]
Final Bias: -2.0

--- Correlation Learning Test ---
Input: [-1 -1], Desired: -1, Net Output: -6.0000
Input: [-1  1], Desired: -1, Net Output: -2.0000
Input: [ 1 -1], Desired: -1, Net Output: -2.0000
Input: [1 1], Desired: 1, Net Output: 2.0000


Outstar Learning Law

The Outstar Learning Law is used when one input neuron is connected to multiple output neurons and we want the output weights to learn a desired output pattern. When an input neuron is active, the connections from that neuron to the output neurons are adjusted toward the corresponding target outputs.

In [41]:
def outstar_learning(inputs, outputs, learning_rate=0.1, epochs=100):
    """
    Demonstrates the Outstar Learning Law.

    Outstar learning updates the weights from an input neuron
    toward multiple output neurons according to:

        delta_w_i = learning_rate * x * (target_i - w_i)

    where:
        x       = input activation
        target  = desired output vector
        w       = current weight vector

    Args:
        inputs (np.array): A 2D array where each row contains the
                           input activation pattern.
        outputs (np.array): A 2D array where each row contains the
                            desired output pattern.
        learning_rate (float): The learning rate (eta).
        epochs (int): Number of training iterations.

    Returns:
        np.array: Final weight matrix.
    """

    num_inputs = inputs.shape[1]
    num_outputs = outputs.shape[1]

    # Outstar connects input neurons to multiple output neurons.
    # Weight matrix shape:
    #     (number of inputs, number of outputs)
    weights = np.zeros((num_inputs, num_outputs))

    num_patterns = inputs.shape[0]

    print("\n--- Outstar Learning Law ---")
    print(f"Initial Weights:\n{weights}")

    for epoch in range(epochs):

        for i in range(num_patterns):

            x = inputs[i]
            target = outputs[i]

            # Outstar update:
            # delta_w_ij = eta * x_i * (target_j - w_ij)
            for j in range(num_outputs):
                weights[:, j] += (
                    learning_rate
                    * x
                    * (target[j] - weights[:, j])
                )

        # Calculate total error only for monitoring
        total_error = 0

        for i in range(num_patterns):

            x = inputs[i]
            target = outputs[i]

            # Linear output of the Outstar
            predicted = x @ weights

            error = target - predicted
            total_error += 0.5 * np.sum(error ** 2)

        print(
            f"Epoch {epoch + 1}: "
            f"Total Squared Error = {total_error:.6f}"
        )

        if total_error < 0.001:
            print(f"Converged at Epoch {epoch + 1}.")
            break

    else:
        print(f"Did not converge within {epochs} epochs.")

    print(f"\nFinal Weights (Outstar):\n{weights}")

    return weights

In [42]:
# Example for Outstar Learning:
# 2 input neurons -> 3 output neurons

outstar_inputs = np.array([
    [1, 0],
    [0, 1]
])

outstar_outputs = np.array([
    [1.0, 0.5, 0.0],   # Desired output for [1, 0]
    [0.0, 0.5, 1.0]    # Desired output for [0, 1]
])

outstar_weights = outstar_learning(
    outstar_inputs,
    outstar_outputs,
    learning_rate=0.5,
    epochs=100
)


--- Outstar Learning Law ---
Initial Weights:
[[0. 0. 0.]
 [0. 0. 0.]]
Epoch 1: Total Squared Error = 0.312500
Epoch 2: Total Squared Error = 0.078125
Epoch 3: Total Squared Error = 0.019531
Epoch 4: Total Squared Error = 0.004883
Epoch 5: Total Squared Error = 0.001221
Epoch 6: Total Squared Error = 0.000305
Converged at Epoch 6.

Final Weights (Outstar):
[[0.984375  0.4921875 0.       ]
 [0.        0.4921875 0.984375 ]]


In [43]:
print("\n--- Outstar Test ---")

for i in range(len(outstar_inputs)):

    x = outstar_inputs[i]

    # Linear output:
    # y = xW
    y_predicted = x @ outstar_weights

    print(f"Input: {x}")
    print(f"Desired Output: {outstar_outputs[i]}")
    print(f"Predicted Output: {y_predicted}")
    print()


--- Outstar Test ---
Input: [1 0]
Desired Output: [1.  0.5 0. ]
Predicted Output: [0.984375  0.4921875 0.       ]

Input: [0 1]
Desired Output: [0.  0.5 1. ]
Predicted Output: [0.        0.4921875 0.984375 ]



In [58]:
print("\n=== Learning Rate = 0.1 ===")
outstar_learning(
    outstar_inputs,
    outstar_outputs,
    learning_rate=0.1,
    epochs=100
)

print("\n=== Learning Rate = 0.5 ===")
outstar_learning(
    outstar_inputs,
    outstar_outputs,
    learning_rate=0.5,
    epochs=100
)

print("\n=== Learning Rate = 1.0 ===")
outstar_learning(
    outstar_inputs,
    outstar_outputs,
    learning_rate=1.0,
    epochs=100
)


=== Learning Rate = 0.1 ===

--- Outstar Learning Law ---
Initial Weights:
[[0. 0. 0.]
 [0. 0. 0.]]
Epoch 1: Total Squared Error = 1.012500
Epoch 2: Total Squared Error = 0.820125
Epoch 3: Total Squared Error = 0.664301
Epoch 4: Total Squared Error = 0.538084
Epoch 5: Total Squared Error = 0.435848
Epoch 6: Total Squared Error = 0.353037
Epoch 7: Total Squared Error = 0.285960
Epoch 8: Total Squared Error = 0.231628
Epoch 9: Total Squared Error = 0.187618
Epoch 10: Total Squared Error = 0.151971
Epoch 11: Total Squared Error = 0.123096
Epoch 12: Total Squared Error = 0.099708
Epoch 13: Total Squared Error = 0.080764
Epoch 14: Total Squared Error = 0.065418
Epoch 15: Total Squared Error = 0.052989
Epoch 16: Total Squared Error = 0.042921
Epoch 17: Total Squared Error = 0.034766
Epoch 18: Total Squared Error = 0.028160
Epoch 19: Total Squared Error = 0.022810
Epoch 20: Total Squared Error = 0.018476
Epoch 21: Total Squared Error = 0.014966
Epoch 22: Total Squared Error = 0.012122
Epoch 

array([[1. , 0.5, 0. ],
       [0. , 0.5, 1. ]])

Instar (Winner-Take-All) Learning Law

The Instar Learning Law is an unsupervised competitive learning rule. Multiple output neurons compete to respond to an input, and only the winner — the neuron with the largest activation — updates its weights.

In [48]:
def instar_learning(inputs, learning_rate=0.1, epochs=100):
    """
    Demonstrates the Instar (Winner-Take-All) Learning Law.

    Instar is an unsupervised competitive learning rule.
    For each input pattern, the output neuron with the highest
    activation wins, and only the winning neuron's weights are updated.

    Rule:
        delta_w_i = learning_rate * x_i * (1 - w_i)

    Args:
        inputs (np.array): A 2D array where each row is an input pattern.
        learning_rate (float): The learning rate (eta).
        epochs (int): Number of training iterations.

    Returns:
        np.array: Final weight matrix.
    """

    num_patterns = inputs.shape[0]
    num_features = inputs.shape[1]

    num_neurons = num_patterns

    # Non-zero initialization is important for competition.
    np.random.seed(42)
    weights = np.random.rand(num_neurons, num_features)

    print("\n--- Instar (Winner-Take-All) Learning Law ---")
    print(f"Initial Weights:\n{weights}")

    for epoch in range(epochs):

        for i in range(num_patterns):

            x = inputs[i]

            # Calculate activation of every neuron
            activations = weights @ x

            # Winner = neuron with maximum activation
            winner = np.argmax(activations)

            # Only winner learns
            delta_weights = (
                learning_rate
                * x
                * (1 - weights[winner])
            )

            weights[winner] += delta_weights

        print(
            f"Epoch {epoch + 1}: "
            f"Weights =\n{weights}"
        )

    print(f"\nFinal Weights (Instar):\n{weights}")

    return weights

In [49]:
# Example for Instar Learning:
# Unsupervised Winner-Take-All learning

instar_inputs = np.array([
    [1.0, 0.0],
    [0.9, 0.1],
    [0.8, 0.2],
    [0.0, 1.0],
    [0.1, 0.9],
    [0.2, 0.8]
])

instar_weights = instar_learning(
    instar_inputs,
    learning_rate=0.2,
    epochs=5
)


--- Instar (Winner-Take-All) Learning Law ---
Initial Weights:
[[0.37454012 0.95071431]
 [0.73199394 0.59865848]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.96990985]]
Epoch 1: Weights =
[[0.41156734 0.96605201]
 [0.85231794 0.6224179 ]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.97592788]]
Epoch 2: Weights =
[[0.44640256 0.97661663]
 [0.91862128 0.64477076]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.98074231]]
Epoch 3: Weights =
[[0.47917553 0.98389353]
 [0.95515707 0.66580033]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.98459384]]
Epoch 4: Weights =
[[0.51000833 0.98890587]
 [0.97528975 0.68558495]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.98767508]]
Epoch 5: Weights =
[[0.53901584 0.99388669]
 [0.98638366 0.70419832]
 [0.15601864 0.15599452]
 [0.0580836

In [50]:
print("\n--- Instar Test ---")

for i in range(len(instar_inputs)):

    x = instar_inputs[i]

    # Calculate activation of each neuron
    activations = instar_weights @ x

    # Winner is the neuron with highest activation
    winner = np.argmax(activations)

    print(f"Input: {x}")
    print(f"Activations: {activations}")
    print(f"Winner: Neuron {winner}")
    print()


--- Instar Test ---
Input: [1. 0.]
Activations: [0.53901584 0.98638366 0.15601864 0.05808361 0.60111501 0.02058449]
Winner: Neuron 1

Input: [0.9 0.1]
Activations: [0.58450293 0.95816513 0.15601623 0.13889287 0.61181077 0.11729355]
Winner: Neuron 1

Input: [0.8 0.2]
Activations: [0.62999001 0.9299466  0.15601382 0.21970212 0.62250652 0.21400261]
Winner: Neuron 1

Input: [0. 1.]
Activations: [0.99388669 0.70419832 0.15599452 0.86617615 0.70807258 0.98767508]
Winner: Neuron 0

Input: [0.1 0.9]
Activations: [0.9483996  0.73241686 0.15599693 0.78536689 0.69737682 0.89096602]
Winner: Neuron 0

Input: [0.2 0.8]
Activations: [0.90291252 0.76063539 0.15599934 0.70455764 0.68668106 0.79425696]
Winner: Neuron 0



In [59]:
print("\n=== Learning Rate = 0.1 ===")
instar_learning(
    instar_inputs,
    learning_rate=0.1,
    epochs=5
)

print("\n=== Learning Rate = 0.5 ===")
instar_learning(
    instar_inputs,
    learning_rate=0.5,
    epochs=5
)

print("\n=== Learning Rate = 1.0 ===")
instar_learning(
    instar_inputs,
    learning_rate=1.0,
    epochs=5
)


=== Learning Rate = 0.1 ===

--- Instar (Winner-Take-All) Learning Law ---
Initial Weights:
[[0.37454012 0.95071431]
 [0.73199394 0.59865848]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.96990985]]
Epoch 1: Weights =
[[0.39317882 0.95873802]
 [0.7980628  0.61061846]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.97291887]]
Epoch 2: Weights =
[[0.41126209 0.96545547]
 [0.84784435 0.62222203]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.97562698]]
Epoch 3: Weights =
[[0.42880648 0.97107932]
 [0.88535376 0.63347981]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.97806428]]
Epoch 4: Weights =
[[0.44582805 0.9757876 ]
 [0.91361635 0.64440212]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]
 [0.02058449 0.98025785]]
Epoch 5: Weights =
[[0.46234237 0.97972938]
 [0.93491165 0.65499893]
 [0.156

array([[0.86553185, 1.        ],
       [1.        , 0.92234372],
       [0.15601864, 0.15599452],
       [0.05808361, 0.86617615],
       [0.60111501, 0.70807258],
       [0.11852604, 1.        ]])

Reinforcement Learning Law

In reinforcement learning, the neuron learns from a reward or punishment rather than being given the correct target output for every input. After producing an output, the environment provides a reinforcement signal r: a positive value strengthens the connection that contributed to the successful response, while a negative value weakens it.

In [51]:
def reinforcement_learning(inputs, rewards, learning_rate=0.1, epochs=100):
    """
    Demonstrates a simple Reinforcement Learning Law using BasicNeuron.

    The weight update rule is:

        delta_w_i = learning_rate * x_i * reward

    The bias is treated as a weight connected to a constant input 1:

        delta_bias = learning_rate * reward

    Args:
        inputs (np.array): A 2D array where each row is an input pattern.
        rewards (np.array): A 1D array containing the reinforcement signal
                            for each input pattern.
                            Positive = reward
                            Negative = punishment
        learning_rate (float): The learning rate (eta).
        epochs (int): Number of training iterations.

    Returns:
        tuple: Final weights and bias.
    """

    num_inputs = inputs.shape[1]

    neuron = BasicNeuron(
        num_inputs,
        activation_function='linear'
    )

    # Initialize weights and bias to zero
    neuron.update_weights(
        np.zeros(num_inputs),
        0.0
    )

    num_patterns = inputs.shape[0]

    print("\n--- Reinforcement Learning ---")
    print(
        f"Initial Weights: {neuron.weights}, "
        f"Initial Bias: {neuron.bias}"
    )

    for epoch in range(epochs):

        for i in range(num_patterns):

            x = inputs[i]
            reward = rewards[i]

            # Reinforcement learning rule
            delta_weights = (
                learning_rate
                * x
                * reward
            )

            # Bias update
            delta_bias = learning_rate * reward

            new_weights = neuron.weights + delta_weights
            new_bias = neuron.bias + delta_bias

            neuron.update_weights(
                new_weights,
                new_bias
            )

        print(
            f"Epoch {epoch + 1}: "
            f"Weights = {neuron.weights}, "
            f"Bias = {neuron.bias:.4f}"
        )

    print(
        f"\nFinal Weights (Reinforcement): "
        f"{neuron.weights}"
    )
    print(f"Final Bias: {neuron.bias}")

    return neuron.weights, neuron.bias

In [52]:
# Example for Reinforcement Learning

reinforcement_inputs = np.array([
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 0]
])

# Positive reward = desirable response
# Negative reward = undesirable response
reinforcement_rewards = np.array([
     1.0,
    -1.0,
     1.0,
    -1.0
])

reinforcement_learning(
    reinforcement_inputs,
    reinforcement_rewards,
    learning_rate=0.1,
    epochs=10
)


--- Reinforcement Learning ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Epoch 1: Weights = [0.2 0. ], Bias = 0.0000
Epoch 2: Weights = [0.4 0. ], Bias = 0.0000
Epoch 3: Weights = [0.6 0. ], Bias = 0.0000
Epoch 4: Weights = [0.8 0. ], Bias = 0.0000
Epoch 5: Weights = [1. 0.], Bias = 0.0000
Epoch 6: Weights = [1.2 0. ], Bias = 0.0000
Epoch 7: Weights = [1.4 0. ], Bias = 0.0000
Epoch 8: Weights = [1.6 0. ], Bias = 0.0000
Epoch 9: Weights = [1.8 0. ], Bias = 0.0000
Epoch 10: Weights = [2. 0.], Bias = 0.0000

Final Weights (Reinforcement): [2. 0.]
Final Bias: 0.0


(array([2., 0.]), np.float64(0.0))

In [53]:
# Train the reinforcement learner
weights, bias = reinforcement_learning(
    reinforcement_inputs,
    reinforcement_rewards,
    learning_rate=0.1,
    epochs=10
)

# Create a neuron for testing
reinforcement_neuron = BasicNeuron(
    2,
    activation_function='linear'
)

reinforcement_neuron.update_weights(
    weights,
    bias
)

print("\n--- Reinforcement Learning Test ---")

for i in range(len(reinforcement_inputs)):

    x = reinforcement_inputs[i]

    y_predicted = reinforcement_neuron.forward(x)

    print(
        f"Input: {x}, "
        f"Reward: {reinforcement_rewards[i]}, "
        f"Output: {y_predicted:.4f}"
    )


--- Reinforcement Learning ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Epoch 1: Weights = [0.2 0. ], Bias = 0.0000
Epoch 2: Weights = [0.4 0. ], Bias = 0.0000
Epoch 3: Weights = [0.6 0. ], Bias = 0.0000
Epoch 4: Weights = [0.8 0. ], Bias = 0.0000
Epoch 5: Weights = [1. 0.], Bias = 0.0000
Epoch 6: Weights = [1.2 0. ], Bias = 0.0000
Epoch 7: Weights = [1.4 0. ], Bias = 0.0000
Epoch 8: Weights = [1.6 0. ], Bias = 0.0000
Epoch 9: Weights = [1.8 0. ], Bias = 0.0000
Epoch 10: Weights = [2. 0.], Bias = 0.0000

Final Weights (Reinforcement): [2. 0.]
Final Bias: 0.0

--- Reinforcement Learning Test ---
Input: [1 0], Reward: 1.0, Output: 2.0000
Input: [0 1], Reward: -1.0, Output: 0.0000
Input: [1 1], Reward: 1.0, Output: 2.0000
Input: [0 0], Reward: -1.0, Output: 0.0000


In [60]:
print("\n=== Learning Rate = 0.01 ===")
reinforcement_learning(
    reinforcement_inputs,
    reinforcement_rewards,
    learning_rate=0.01,
    epochs=10
)

print("\n=== Learning Rate = 0.1 ===")
reinforcement_learning(
    reinforcement_inputs,
    reinforcement_rewards,
    learning_rate=0.1,
    epochs=10
)

print("\n=== Learning Rate = 0.5 ===")
reinforcement_learning(
    reinforcement_inputs,
    reinforcement_rewards,
    learning_rate=0.5,
    epochs=10
)


=== Learning Rate = 0.01 ===

--- Reinforcement Learning ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Epoch 1: Weights = [0.02 0.  ], Bias = 0.0000
Epoch 2: Weights = [0.04 0.  ], Bias = 0.0000
Epoch 3: Weights = [0.06 0.  ], Bias = 0.0000
Epoch 4: Weights = [0.08 0.  ], Bias = 0.0000
Epoch 5: Weights = [0.1 0. ], Bias = 0.0000
Epoch 6: Weights = [0.12 0.  ], Bias = 0.0000
Epoch 7: Weights = [0.14 0.  ], Bias = 0.0000
Epoch 8: Weights = [0.16 0.  ], Bias = 0.0000
Epoch 9: Weights = [0.18 0.  ], Bias = 0.0000
Epoch 10: Weights = [0.2 0. ], Bias = 0.0000

Final Weights (Reinforcement): [0.2 0. ]
Final Bias: 0.0

=== Learning Rate = 0.1 ===

--- Reinforcement Learning ---
Initial Weights: [0. 0.], Initial Bias: 0.0
Epoch 1: Weights = [0.2 0. ], Bias = 0.0000
Epoch 2: Weights = [0.4 0. ], Bias = 0.0000
Epoch 3: Weights = [0.6 0. ], Bias = 0.0000
Epoch 4: Weights = [0.8 0. ], Bias = 0.0000
Epoch 5: Weights = [1. 0.], Bias = 0.0000
Epoch 6: Weights = [1.2 0. ], Bias = 0.0000
Epoch 7: Wei

(array([10.,  0.]), np.float64(0.0))

Boltzmann Learning Law

Boltzmann Learning is a probabilistic learning rule used in stochastic neural networks such as Boltzmann Machines. Unlike rules such as Hebbian or Delta learning, the network does not simply deterministically update a weight from one input-output pair.

In [54]:
def boltzmann_learning(inputs, learning_rate=0.1, epochs=100, k=10):
    """
    Demonstrates a simplified Boltzmann Learning Law.

    The weight update is based on the difference between:
        P_data  = correlation observed in the training data
        P_model = correlation observed from the model's samples

    Rule:
        delta_w = learning_rate * (P_data - P_model)

    A simplified stochastic binary neuron model is used here.

    Args:
        inputs (np.array): A 2D array where each row is a binary
                           training pattern.
        learning_rate (float): Learning rate (eta).
        epochs (int): Number of training iterations.
        k (int): Number of samples used to estimate model correlations.

    Returns:
        np.array: Learned weight matrix.
    """

    num_features = inputs.shape[1]

    # Initialize weights to zero
    weights = np.zeros((num_features, num_features))

    print("\n--- Boltzmann Learning Law ---")
    print(f"Initial Weights:\n{weights}")

    # Calculate data correlations.
    # Each element represents how frequently two features
    # are simultaneously active in the training data.
    p_data = (inputs.T @ inputs) / inputs.shape[0]

    for epoch in range(epochs):

        # ---------------------------------------------------------
        # Generate samples from the current model
        # ---------------------------------------------------------

        model_states = []

        for _ in range(k):

            # Start with a random binary state
            state = np.random.randint(
                0, 2, size=num_features
            ).astype(float)

            # One stochastic update of every neuron
            for i in range(num_features):

                # Calculate activation / net input
                net_input = np.dot(
                    weights[i],
                    state
                )

                # Sigmoid gives probability of neuron being active
                probability = 1 / (1 + np.exp(-net_input))

                # Sample neuron state probabilistically
                state[i] = (
                    1
                    if np.random.rand() < probability
                    else 0
                )

            model_states.append(state)

        model_states = np.array(model_states)

        # Model correlations
        p_model = (
            model_states.T @ model_states
        ) / k

        # ---------------------------------------------------------
        # Boltzmann Learning Rule
        # ---------------------------------------------------------

        delta_weights = (
            learning_rate
            * (p_data - p_model)
        )

        # Update weights
        weights += delta_weights

        # Keep the matrix symmetric
        weights = (weights + weights.T) / 2

        # No self-connections
        np.fill_diagonal(weights, 0)

        print(
            f"Epoch {epoch + 1}: "
            f"Weight Change = "
            f"{np.linalg.norm(delta_weights):.6f}"
        )

        # Stop when the update becomes very small
        if np.linalg.norm(delta_weights) < 0.001:
            print(f"Converged at Epoch {epoch + 1}.")
            break

    else:
        print(f"Did not converge within {epochs} epochs.")

    print(f"\nFinal Weights (Boltzmann):\n{weights}")

    return weights

In [55]:
# Example for Boltzmann Learning

boltzmann_inputs = np.array([
    [0, 0],
    [0, 0],
    [1, 1],
    [1, 1],
    [1, 1],
    [0, 0]
])

boltzmann_weights = boltzmann_learning(
    boltzmann_inputs,
    learning_rate=0.1,
    epochs=100,
    k=50
)


--- Boltzmann Learning Law ---
Initial Weights:
[[0. 0.]
 [0. 0.]]
Epoch 1: Weight Change = 0.034176
Epoch 2: Weight Change = 0.050120
Epoch 3: Weight Change = 0.030133
Epoch 4: Weight Change = 0.031113
Epoch 5: Weight Change = 0.017889
Epoch 6: Weight Change = 0.027641
Epoch 7: Weight Change = 0.034699
Epoch 8: Weight Change = 0.034986
Epoch 9: Weight Change = 0.039850
Epoch 10: Weight Change = 0.035100
Epoch 11: Weight Change = 0.045695
Epoch 12: Weight Change = 0.026683
Epoch 13: Weight Change = 0.034525
Epoch 14: Weight Change = 0.026683
Epoch 15: Weight Change = 0.046819
Epoch 16: Weight Change = 0.019799
Epoch 17: Weight Change = 0.038053
Epoch 18: Weight Change = 0.026758
Epoch 19: Weight Change = 0.021071
Epoch 20: Weight Change = 0.040447
Epoch 21: Weight Change = 0.031623
Epoch 22: Weight Change = 0.020785
Epoch 23: Weight Change = 0.028844
Epoch 24: Weight Change = 0.023495
Epoch 25: Weight Change = 0.018221
Epoch 26: Weight Change = 0.020591
Epoch 27: Weight Change = 0.021

In [56]:
def generate_boltzmann_sample(weights, steps=100):
    """
    Generate a binary sample from a simplified
    Boltzmann model using stochastic updates.
    """

    num_features = weights.shape[0]

    state = np.random.randint(
        0, 2, size=num_features
    ).astype(float)

    for _ in range(steps):

        for i in range(num_features):

            net_input = np.dot(
                weights[i],
                state
            )

            probability = 1 / (1 + np.exp(-net_input))

            state[i] = (
                1
                if np.random.rand() < probability
                else 0
            )

    return state


print("\n--- Boltzmann Model Test ---")

for i in range(10):

    sample = generate_boltzmann_sample(
        boltzmann_weights
    )

    print(f"Generated Sample {i + 1}: {sample}")


--- Boltzmann Model Test ---
Generated Sample 1: [0. 0.]
Generated Sample 2: [1. 1.]
Generated Sample 3: [0. 0.]
Generated Sample 4: [0. 1.]
Generated Sample 5: [0. 0.]
Generated Sample 6: [0. 1.]
Generated Sample 7: [1. 1.]
Generated Sample 8: [1. 1.]
Generated Sample 9: [1. 1.]
Generated Sample 10: [1. 1.]


In [57]:
print("\n=== Learning Rate = 0.01 ===")
boltzmann_learning(
    boltzmann_inputs,
    learning_rate=0.01,
    epochs=100,
    k=50
)

print("\n=== Learning Rate = 0.1 ===")
boltzmann_learning(
    boltzmann_inputs,
    learning_rate=0.1,
    epochs=100,
    k=50
)

print("\n=== Learning Rate = 0.5 ===")
boltzmann_learning(
    boltzmann_inputs,
    learning_rate=0.5,
    epochs=100,
    k=50
)


=== Learning Rate = 0.01 ===

--- Boltzmann Learning Law ---
Initial Weights:
[[0. 0.]
 [0. 0.]]
Epoch 1: Weight Change = 0.005415
Epoch 2: Weight Change = 0.002919
Epoch 3: Weight Change = 0.003688
Epoch 4: Weight Change = 0.004891
Epoch 5: Weight Change = 0.004850
Epoch 6: Weight Change = 0.002843
Epoch 7: Weight Change = 0.004907
Epoch 8: Weight Change = 0.002585
Epoch 9: Weight Change = 0.003098
Epoch 10: Weight Change = 0.004414
Epoch 11: Weight Change = 0.005284
Epoch 12: Weight Change = 0.003811
Epoch 13: Weight Change = 0.004303
Epoch 14: Weight Change = 0.002209
Epoch 15: Weight Change = 0.003985
Epoch 16: Weight Change = 0.004750
Epoch 17: Weight Change = 0.003704
Epoch 18: Weight Change = 0.002585
Epoch 19: Weight Change = 0.003970
Epoch 20: Weight Change = 0.004600
Epoch 21: Weight Change = 0.003965
Epoch 22: Weight Change = 0.004290
Epoch 23: Weight Change = 0.002919
Epoch 24: Weight Change = 0.003423
Epoch 25: Weight Change = 0.002857
Epoch 26: Weight Change = 0.005250
E

array([[0.  , 1.23],
       [1.23, 0.  ]])

Widrow-Hoff LMS Learning Law

The Widrow-Hoff Learning Law, also called the Least Mean Squares (LMS) Rule, is a supervised learning rule used by ADALINE. It updates the weights according to the error between the desired output and the neuron's actual linear output. The important point is that, unlike the Perceptron, LMS uses the continuous linear output before applying any threshold

In [61]:
def widrow_hoff_learning(inputs, outputs, learning_rate=0.01, epochs=100, activation_func='linear'):
    """
    Demonstrates the Widrow-Hoff (LMS) Learning Law using BasicNeuron.

    The LMS update rule is:

        delta_w_i = learning_rate * (target - output) * x_i

    and the bias update is:

        delta_bias = learning_rate * (target - output)

    The output used for learning is the linear output of ADALINE.

    Args:
        inputs (np.array): A 2D array where each row is an input pattern.
        outputs (np.array): A 1D array of desired/target outputs.
        learning_rate (float): The learning rate (eta).
        epochs (int): Number of training iterations.
        activation_func (str): Normally 'linear' for ADALINE.

    Returns:
        tuple: Final weights and bias.
    """

    num_inputs = inputs.shape[1]

    # ADALINE uses a linear activation during learning
    neuron = BasicNeuron(
        num_inputs,
        activation_function=activation_func
    )

    num_patterns = inputs.shape[0]

    print(
        f"\n--- Widrow-Hoff LMS Learning "
        f"({activation_func.capitalize()} Activation) ---"
    )
    print(
        f"Initial Weights: {neuron.weights}, "
        f"Initial Bias: {neuron.bias}"
    )

    for epoch in range(epochs):

        total_error = 0

        for i in range(num_patterns):

            x = inputs[i]
            y_desired = outputs[i]

            # ADALINE produces a continuous linear output
            y_actual = neuron.forward(x)

            # Error
            error = y_desired - y_actual

            # Accumulate squared error
            total_error += 0.5 * (error ** 2)

            # Widrow-Hoff / LMS update
            delta_weights = (
                learning_rate
                * error
                * x
            )

            delta_bias = learning_rate * error

            # Update weights and bias
            new_weights = neuron.weights + delta_weights
            new_bias = neuron.bias + delta_bias

            neuron.update_weights(
                new_weights,
                new_bias
            )

        print(
            f"Epoch {epoch + 1}: "
            f"Total Squared Error = {total_error:.6f}, "
            f"Weights = {neuron.weights}, "
            f"Bias = {neuron.bias:.6f}"
        )

        # Stop when error becomes sufficiently small
        if total_error < 0.001:
            print(
                f"Converged at Epoch {epoch + 1}."
            )
            break

    else:
        print(
            f"Did not converge within {epochs} epochs."
        )

    print(
        f"\nFinal Weights (Widrow-Hoff LMS): "
        f"{neuron.weights}"
    )
    print(f"Final Bias: {neuron.bias}")

    return neuron.weights, neuron.bias

In [62]:
# Example for Widrow-Hoff LMS:
# Learn approximately y = 2x + 1

lms_inputs = np.array([
    [0.1],
    [0.2],
    [0.3],
    [0.4],
    [0.5],
    [0.6],
    [0.7],
    [0.8],
    [0.9],
    [1.0]
])

lms_outputs = np.array([
    1.2,
    1.4,
    1.6,
    1.8,
    2.0,
    2.2,
    2.4,
    2.6,
    2.8,
    3.0
])

lms_weights, lms_bias = widrow_hoff_learning(
    lms_inputs,
    lms_outputs,
    learning_rate=0.01,
    epochs=1000,
    activation_func='linear'
)


--- Widrow-Hoff LMS Learning (Linear Activation) ---
Initial Weights: [-0.72101228], Initial Bias: -0.4157107029295637
Epoch 1: Total Squared Error = 40.601828, Weights = [-0.54950235], Bias = -0.138652
Epoch 2: Total Squared Error = 31.246933, Weights = [-0.39833675], Bias = 0.103215
Epoch 3: Total Squared Error = 24.084149, Weights = [-0.26498415], Bias = 0.314285
Epoch 4: Total Squared Error = 18.599907, Weights = [-0.14722897], Bias = 0.498404
Epoch 5: Total Squared Error = 14.400885, Weights = [-0.04313195], Bias = 0.658937
Epoch 6: Total Squared Error = 11.185859, Weights = [0.04900428], Bias = 0.798831
Epoch 7: Total Squared Error = 8.724148, Weights = [0.13066537], Bias = 0.920664
Epoch 8: Total Squared Error = 6.839105, Weights = [0.2031517], Bias = 1.026693
Epoch 9: Total Squared Error = 5.395461, Weights = [0.26760147], Bias = 1.118893
Epoch 10: Total Squared Error = 4.289646, Weights = [0.32501092], Bias = 1.198994
Epoch 11: Total Squared Error = 3.442360, Weights = [0.376

In [63]:
# Create a neuron with the learned parameters

lms_neuron = BasicNeuron(
    1,
    activation_function='linear'
)

lms_neuron.update_weights(
    lms_weights,
    lms_bias
)

print("\n--- Widrow-Hoff LMS Test ---")

for i in range(len(lms_inputs)):

    x = lms_inputs[i]

    y_predicted = lms_neuron.forward(x)

    print(
        f"Input: {x}, "
        f"Desired: {lms_outputs[i]:.2f}, "
        f"Predicted: {y_predicted:.4f}"
    )


--- Widrow-Hoff LMS Test ---
Input: [0.1], Desired: 1.20, Predicted: 1.2239
Input: [0.2], Desired: 1.40, Predicted: 1.4191
Input: [0.3], Desired: 1.60, Predicted: 1.6143
Input: [0.4], Desired: 1.80, Predicted: 1.8095
Input: [0.5], Desired: 2.00, Predicted: 2.0047
Input: [0.6], Desired: 2.20, Predicted: 2.1999
Input: [0.7], Desired: 2.40, Predicted: 2.3951
Input: [0.8], Desired: 2.60, Predicted: 2.5903
Input: [0.9], Desired: 2.80, Predicted: 2.7855
Input: [1.], Desired: 3.00, Predicted: 2.9807


In [64]:
print("\n=== Learning Rate = 0.001 ===")
widrow_hoff_learning(
    lms_inputs,
    lms_outputs,
    learning_rate=0.001,
    epochs=1000,
    activation_func='linear'
)

print("\n=== Learning Rate = 0.01 ===")
widrow_hoff_learning(
    lms_inputs,
    lms_outputs,
    learning_rate=0.01,
    epochs=1000,
    activation_func='linear'
)

print("\n=== Learning Rate = 0.1 ===")
widrow_hoff_learning(
    lms_inputs,
    lms_outputs,
    learning_rate=0.1,
    epochs=1000,
    activation_func='linear'
)


=== Learning Rate = 0.001 ===

--- Widrow-Hoff LMS Learning (Linear Activation) ---
Initial Weights: [0.57035192], Initial Bias: -0.6006524356832805
Epoch 1: Total Squared Error = 28.990032, Weights = [0.58456169], Bias = -0.576909
Epoch 2: Total Squared Error = 28.233962, Weights = [0.59858749], Bias = -0.553480
Epoch 3: Total Squared Error = 27.497643, Weights = [0.61243172], Bias = -0.530360
Epoch 4: Total Squared Error = 26.780561, Weights = [0.62609678], Bias = -0.507546
Epoch 5: Total Squared Error = 26.082212, Weights = [0.63958502], Bias = -0.485034
Epoch 6: Total Squared Error = 25.402107, Weights = [0.65289876], Bias = -0.462819
Epoch 7: Total Squared Error = 24.739769, Weights = [0.66604029], Bias = -0.440898
Epoch 8: Total Squared Error = 24.094734, Weights = [0.67901189], Bias = -0.419267
Epoch 9: Total Squared Error = 23.466550, Weights = [0.69181577], Bias = -0.397923
Epoch 10: Total Squared Error = 22.854778, Weights = [0.70445415], Bias = -0.376860
Epoch 11: Total Squ

(array([1.95950805]), np.float64(1.0286561060903323))